# image_lab in a notebook (T016)

Minimal portable notebook over the gallery program
`examples/gallery/image_lab.jl` (import-safe: including it renders
nothing). Requires only Vetra + PNGFiles in an isolated project —
setup in `README.md`, section *VS Code / Jupyter notebooks*.

The contract asserts below are hand-derived; a failure errors the
cell (no silent skip, no tolerance). The presentation cells use
image functions only — no font fixtures needed.


In [ ]:
using Vetra

In [ ]:
# package-derived location — no host-specific absolute path
include(joinpath(pkgdir(Vetra), "examples", "gallery", "image_lab.jl"))

In [ ]:
img = test_image()  # 4x4, every cell a distinct valid premultiplied pixel

# --- MIME byte identity + source preservation (hand-derived) ---
snap = copy(img.data)
buf = IOBuffer()
show(buf, MIME"image/png"(), img)
mime = take!(buf)  # public API: Vector{UInt8} on 1.10 and 1.13
@assert mime == encode_png(img) "MIME payload is not the encode_png bytes"
@assert img.data == snap        "displaying the image mutated the source"

# --- independent pixels (valid premul storage, x/y per get_color) ---
@assert get_color(img, 1, 1) == rgbx(255, 0, 0, 255)     "(1,1) is not premul red"
@assert get_color(img, 4, 1) == rgbx(128, 128, 128, 128) "(4,1) is not premul of straight (255,255,255,128)"
@assert get_color(img, 4, 4) == rgbx(0, 0, 0, 255)       "(4,4) is not opaque black"

# --- round trip through the file boundary (premul storage restored) ---
@assert decode_png(mime).data == img.data "PNG round trip is not byte-exact"

In [ ]:
big = enlarge8(img)  # 32x32, byte-exact 8x (integer scale: pixel centers)
@assert size(big) == (32, 32)
@assert get_color(big, 4, 4) == rgbx(255, 0, 0, 255) "enlarge8 changed the red block"
big  # bare display -> show(MIME"image/png", big)

In [ ]:
# before/after strip: original / inverted / 50% opacity / rotated 90
# (font-free: the gallery's composite() also renders text and needs the
# gitignored .reference font fixtures; here only image functions)
snap2 = copy(img.data)  # the panel effects must not touch the source
panels = (img,
    (a = copy(img); invert!(a); a),
    (b = copy(img); apply_opacity!(b, 0.5f0); b),
    rotate_90(img))
canvas = new_image(200, 64)
ctx = Context(canvas)
ctx.fillStyle = rgb(248, 250, 252)
fill_rect!(ctx, 0f0, 0f0, 200f0, 64f0)
for (i, p) in enumerate(panels)
    draw_image!(ctx, enlarge8(p), 16f0 + 48f0 * (i - 1), 16f0)
end
@assert size(canvas) == (64, 200) "strip canvas is not 200x64 (size is matrix order: 1 = height, 2 = width)"
@assert img.data == snap2 "building the panels mutated the source image"
@assert get_color(canvas, 5, 3) == rgbx(248, 250, 252, 255) "background fill is not opaque rgb(248,250,252)"
canvas  # bare display -> show(MIME"image/png", canvas)